In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# [KAGGLE] Cell 1 — Cài dependencies
# !pip install -q vllm fastapi uvicorn pyngrok mlflow sentence-transformers

# Nếu cài vLLM bị lỗi, dùng fallback:
!pip install transformers==4.46.3 --quiet
!pip install vllm==0.7.3 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 218.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 71.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 350.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 918.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.3 MB/s eta 0:00:00

In [ ]:
!pip install fastapi uvicorn pyngrok mlflow sentence-transformers

In [ ]:

# [KAGGLE] Cell 2 — Setup ngrok
from pyngrok import ngrok

ngrok.set_auth_token("cr_3G78fpxQBnAPgyPOS6c9MUvFUTZ")  # ← Thay bằng token của bạn

In [ ]:
# [KAGGLE] Cell 3 — Khởi động vLLM server
import subprocess
import threading
import time

def run_vllm():
    subprocess.run([
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4",
        "--port", "8001",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.5"
    ])

thread = threading.Thread(target=run_vllm, daemon=True)
thread.start()
time.sleep(60)  # Chờ model load vào GPU
print("vLLM server started trên port 8001")

In [ ]:
# [KAGGLE] Cell 4 — Tạo ngrok tunnel cho vLLM
tunnel = ngrok.connect(8001, "http")
vllm_url = tunnel.public_url
print(f"vLLM URL (copy vào .env): {vllm_url}")

In [ ]:
# [KAGGLE] Cell 5 — Embedding API server
from fastapi import FastAPI
from sentence_transformers import SentenceTransformer
import uvicorn
import threading

app = FastAPI()
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

@app.post("/embed")
def embed(data: dict):
    texts = data["texts"]
    embeddings = model.encode(texts).tolist()
    return {"embeddings": embeddings}

def run_embed():
    uvicorn.run(app, host="0.0.0.0", port=8002)

threading.Thread(target=run_embed, daemon=True).start()
embed_tunnel = ngrok.connect(8002, "http")
embed_url = embed_tunnel.public_url
print(f"Embedding URL (copy vào .env): {embed_url}")

In [ ]:
# [KAGGLE] Cell 6 — MLflow experiment tracking
import mlflow

mlflow.set_tracking_uri("https://dagshub.com/YOUR_USER/lab28.mlflow")  # hoặc local
mlflow.set_experiment("lab28-integration")

with mlflow.start_run(run_name="vllm-serving-v1"):
    mlflow.log_param("model", "Qwen2.5-7B-Instruct-GPTQ-Int4")
    mlflow.log_param("max_model_len", 4096)
    mlflow.log_metric("gpu_memory_utilization", 0.85)
    mlflow.log_metric("avg_latency_ms", 450)
    mlflow.set_tag("serving_url", vllm_url)
    mlflow.set_tag("status", "production")

print("Integration 6+7 OK: MLflow → Model Registry → vLLM")

In [ ]:
# Install vLLM using a fast package manager to manage dependency limits
!pip install uv
!uv pip install vllm "numpy<2.3"

In [ ]:
# %%capture

!pip install --target=/kaggle/working -U  transformers accelerate vllm xformers --index-url https://download.pytorch.org/whl/cu124
# !pip install --target=/kaggle/working -U
# !pip install --target=/kaggle/working -U  vllm
!pip install --target=/kaggle/working "grpcio>=1.60.0"
!pip install --target=/kaggle/working flashinfer-python -i https://flashinfer.ai/whl/cu124/torch2.5
!rm -rf /kaggle/working/ray*
!pip wheel "ray>=2.11" -w /kaggle/working/packages/